# Demucs Voice Isolation - Pitt数据集

In [ ]:
import time
import warnings
from pathlib import Path
from tqdm import tqdm

import torch
import torchaudio

# 过滤 torchaudio 的弃用警告（不影响功能）
warnings.filterwarnings('ignore', category=UserWarning, module='torchaudio')

print(f"PyTorch版本: {torch.__version__}")
print(f"torchaudio版本: {torchaudio.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
print(f"MPS可用: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

In [ ]:
input_dir = Path('data/processed/Pitt-vocals-demucs')
output_dir = Path('data/processed/Pitt-vocals-demucs-2')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

## Load Demucs Model

In [ ]:
from demucs.pretrained import get_model
from demucs.apply import apply_model

model = get_model('htdemucs')
model.to(device)
model.eval()

## 2 Extract Voice Function

In [ ]:
def extract_vocals(audio_path, model, device):
    """
    使用Demucs从音频中提取语音内容
    
    注意: 对于纯语音录音（无音乐背景），Demucs会将语音归类到'other' stem
    因为'vocals' stem在Demucs中主要指唱歌，而不是说话
   
    """
    # 加载音频
    waveform, sr = torchaudio.load(str(audio_path))
    
    # 确保是双声道（Demucs要求）
    if waveform.shape[0] == 1:
        # 单声道转双声道
        waveform = waveform.repeat(2, 1)
    elif waveform.shape[0] > 2:
        # 多声道取前两个
        waveform = waveform[:2, :]
    
    # 转移到设备
    waveform = waveform.to(device)
    
    # 应用Demucs模型
    # 输入形状: [batch=1, channels=2, samples]
    # 输出形状: [batch=1, stems=4, channels=2, samples]
    with torch.no_grad():
        sources = apply_model(model, waveform[None], device=device)
    
    # ⚠️ 重要: 对于纯语音录音，提取'other' stem (索引3)而不是'vocals' stem (索引0)
    # stems顺序: [vocals=0, drums=1, bass=2, other=3]
    # 'vocals'指唱歌，'other'包含纯语音对话
    speech = sources[0, 3]  # [channels=2, samples] - 提取'other' stem
    
    # 转为单声道（平均两个声道）
    speech_mono = torch.mean(speech, dim=0, keepdim=True)
    
    return speech_mono.cpu(), sr

## 3 Batch process function

In [ ]:
def batch_extract_vocals(files, output_subdir, model, device, group_name):
    """
    批量提取vocals stem
    """
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    for audio_file in tqdm(files, desc=f"Extract {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            continue
        
        try:
            # 提取vocals
            vocals, sr = extract_vocals(audio_file, model, device)
            
            # 保存
            torchaudio.save(str(output_file), vocals, sr, bits_per_sample=16)
            
        except Exception as e:
            print(f"\n Failed: {audio_file.name}: {e}")

## 4 Process

In [2]:
# Process Dementia
dementia_result = batch_extract_vocals(
    dementia_files,
    output_dir / 'Dementia',
    model,
    device,
    'Dementia'
)

# ProcessControl
control_result = batch_extract_vocals(
    control_files,
    output_dir / 'Control',
    model,
    device,
    'Control'
)



NameError: name 'batch_extract_vocals' is not defined